# Quantum Security Comparison Dashboard

Five charts comparing RSA-2048 vs. PQC algorithm families across security posture,
performance, quantum threat scaling, attack time, and payment score heatmaps.
All data comes directly from the existing `algorithms` and `benchmarks` modules.

In [ ]:
from pathlib import Path
import importlib, sys, math, base64
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

project_root = Path.cwd()
sys.path.insert(0, str(project_root / 'src') if (project_root / 'src').exists() else str(project_root))

from algorithms.crypto_algorithms import ALGORITHM_PROFILES, score_encrypted_transaction
import algorithms.quantum_auditor as _qa
_qa = importlib.reload(_qa)
import benchmarks.algorithm_benchmark as _bm
_bm = importlib.reload(_bm)

ALGORITHMS   = list(ALGORITHM_PROFILES.keys())
ATTACK_PROF  = _bm.ATTACK_PROFILES
RISK_COLORS  = {'LOW':'#157347','MEDIUM':'#997404','HIGH':'#b54708','CRITICAL':'#b42318'}
ALGO_COLORS  = {
    'RSA-2048':'#b42318', 'ML-KEM-768':'#157347',
    'ML-DSA-65':'#1c6dd0', 'SLH-DSA-128s':'#7c3aed', 'HQC-128':'#b45309',
}
print('Algorithms:', ALGORITHMS)

## Section 1: Algorithm Comparison Radar Chart

In [ ]:
bench = _bm.benchmark_results_as_dicts(iterations=3)
perf  = {r['algorithm']: r['avg_ms'] for r in bench}
max_ms = max(perf.values()) or 1

axes_labels = ['Base Score', 'Quantum Safe', 'Security Bits', 'Performance']
angles = np.linspace(0, 2*np.pi, len(axes_labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6,6), subplot_kw={'polar': True})
ax.set_facecolor('#f8fafc'); fig.patch.set_facecolor('#f8fafc')
ax.set_xticks(angles[:-1]); ax.set_xticklabels(axes_labels, fontsize=11)
ax.set_ylim(0,1); ax.set_yticks([0.25,0.5,0.75,1])
ax.set_yticklabels(['0.25','0.50','0.75','1.00'], fontsize=7, color='#475467')
ax.grid(color='#e4e7ec', linestyle='--', lw=0.8)

for alg, prof in ALGORITHM_PROFILES.items():
    q_bits = ATTACK_PROF.get(alg, {}).get('quantum_security_bits', 0)
    vals = [
        prof.base_score / 100,
        1.0 if prof.quantum_safe else 0.0,
        q_bits / 128,
        1 / (1 + perf.get(alg, max_ms) / max_ms),
    ] + [prof.base_score / 100]  # close polygon
    c = ALGO_COLORS[alg]
    ax.plot(angles, vals, color=c, lw=2, label=alg)
    ax.fill(angles, vals, color=c, alpha=0.10)

ax.legend(loc='upper right', bbox_to_anchor=(1.38,1.15), fontsize=9)
ax.set_title('Algorithm Radar: RSA vs PQC', fontsize=13, pad=18)
plt.tight_layout(); plt.show()

## Section 2: Encryption / KEM / Signature Time Comparison

In [ ]:
bench5 = _bm.benchmark_results_as_dicts(iterations=5)
algs   = [r['algorithm'] for r in bench5]
avg_ms = [r['avg_ms']    for r in bench5]
p95_ms = [r['p95_ms']    for r in bench5]
colors = [ALGO_COLORS[a] for a in algs]
x, w   = np.arange(len(algs)), 0.35

fig, ax = plt.subplots(figsize=(9,5))
ax.set_facecolor('#f8fafc'); fig.patch.set_facecolor('#f8fafc')
b1 = ax.bar(x-w/2, avg_ms, w, color=colors, alpha=0.85, label='Avg ms')
b2 = ax.bar(x+w/2, p95_ms, w, color=colors, alpha=0.45, hatch='//', label='P95 ms')
ax.set_xticks(x); ax.set_xticklabels(algs, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('Time (ms)'); ax.legend()
ax.set_title('Operation Time — Avg vs P95 (ms)', fontsize=12)
ax.grid(axis='y', color='#e4e7ec', ls='--')
for b in b1:
    h = b.get_height()
    ax.annotate(f'{h:.2f}', xy=(b.get_x()+b.get_width()/2, h),
                xytext=(0,3), textcoords='offset points', ha='center', fontsize=7)
plt.tight_layout(); plt.show()

## Section 3: Quantum Circuit Depth vs RSA Key Size

In [ ]:
key_sizes = [512, 1024, 2048, 4096]
reports   = [_qa.estimate_quantum_risk(kb) for kb in key_sizes]
gdepths   = [r['gate_depth_real']     for r in reports]
lqubits   = [r['logical_qubits_real'] for r in reports]
pqubits   = [r['physical_qubits_real'] for r in reports]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))
fig.patch.set_facecolor('#f8fafc')

ax1.set_facecolor('#f8fafc')
ax1.plot(key_sizes, gdepths, 'o-', color='#b42318', lw=2.5, ms=8, label='Gate depth (key³)')
for x,y in zip(key_sizes, gdepths):
    ax1.annotate(f'{y:,}', xy=(x,y), xytext=(0,7), textcoords='offset points',
                 ha='center', fontsize=8, color='#b42318')
ax1.set_yscale('log'); ax1.set_xlabel('RSA Key Size (bits)'); ax1.set_ylabel('Gate Depth')
ax1.set_title("Shor's Gate Depth — O(n³) Scaling", fontsize=11)
ax1.grid(color='#e4e7ec', ls='--'); ax1.legend()

ax2.set_facecolor('#f8fafc')
ax2.plot(key_sizes, lqubits, 's--', color='#1c6dd0', lw=2, ms=8, label='Logical qubits (2n)')
ax2t = ax2.twinx()
ax2t.plot(key_sizes, pqubits, '^:', color='#7c3aed', lw=2, ms=8, label='Physical qubits (×1000)')
ax2.set_xlabel('RSA Key Size (bits)'); ax2.set_ylabel('Logical Qubits', color='#1c6dd0')
ax2t.set_ylabel('Physical Qubits', color='#7c3aed')
ax2.set_title('Qubit Requirements vs Key Size', fontsize=11)
ax2.grid(color='#e4e7ec', ls='--')
lines = ax2.get_legend_handles_labels()[0] + ax2t.get_legend_handles_labels()[0]
labs  = ax2.get_legend_handles_labels()[1] + ax2t.get_legend_handles_labels()[1]
ax2.legend(lines, labs, fontsize=8)

plt.tight_layout(); plt.show()

## Section 4: Classical vs Quantum Attack Time

In [ ]:
SEC_PER_YEAR = 365.25*24*3600
OPS_PER_SEC  = 1e9
estimates    = _bm.attack_estimates_as_dicts()

labels, c_log, q_log = [], [], []
for e in estimates:
    ap   = ATTACK_PROF[e['algorithm']]
    c_yr = (2**(ap['classical_security_bits']-1)) / OPS_PER_SEC / SEC_PER_YEAR
    if e['algorithm'] == 'RSA-2048':
        q_yr = float(ap['quantum_years'])
    else:
        qb   = ap.get('quantum_security_bits', 0)
        q_yr = (2**(qb-1)) / OPS_PER_SEC / SEC_PER_YEAR if qb else 0.01
    labels.append(e['algorithm'])
    c_log.append(math.log10(max(c_yr, 1e-3)))
    q_log.append(math.log10(max(q_yr, 1e-3)))

x, w = np.arange(len(labels)), 0.35
fig, ax = plt.subplots(figsize=(10,6))
ax.set_facecolor('#f8fafc'); fig.patch.set_facecolor('#f8fafc')
bc = ax.bar(x-w/2, c_log, w, color=[ALGO_COLORS[l] for l in labels], alpha=0.85, label='Classical')
bq = ax.bar(x+w/2, q_log, w, color='#60a5fa', alpha=0.85, label='Quantum')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('log₁₀(years to break)'); ax.legend(fontsize=10)
ax.set_title('Classical vs Quantum Attack Time (log scale; higher = harder)', fontsize=12)
ax.grid(axis='y', color='#e4e7ec', ls='--')
for b, e in zip(bc, estimates):
    ax.annotate(e['estimated_classical_attack_years'].replace(' years','y'),
                xy=(b.get_x()+b.get_width()/2, b.get_height()),
                xytext=(0,3), textcoords='offset points', ha='center', fontsize=6)
for b, e in zip(bq, estimates):
    ax.annotate(e['estimated_quantum_attack_years'].replace(' years','y'),
                xy=(b.get_x()+b.get_width()/2, b.get_height()),
                xytext=(0,3), textcoords='offset points', ha='center', fontsize=6, color='#1e40af')
plt.tight_layout(); plt.show()

## Section 5: Security Score Heatmap (Algorithm × Channel & Amount)

In [ ]:
CHANNELS  = ['SWIFT_MT103','ACH_CREDIT','CARD','WIRE','PAYMENT']
AMOUNTS   = [10_000, 100_000, 500_000, 1_000_000, 5_000_000]
AMT_LABS  = ['10k','100k','500k','1M','5M']
PAYLOAD   = base64.b64encode(b'quantum-comparison-demo-payload-for-heatmap-1234567890abcdef').decode()

m_ch = np.array([[score_encrypted_transaction(PAYLOAD,a,500_000,ch)['score']
                  for ch in CHANNELS] for a in ALGORITHMS])
m_am = np.array([[score_encrypted_transaction(PAYLOAD,a,amt,'SWIFT_MT103')['score']
                  for amt in AMOUNTS] for a in ALGORITHMS])

risk_cmap = LinearSegmentedColormap.from_list('risk', ['#b42318','#b54708','#997404','#157347'], N=256)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))
fig.patch.set_facecolor('#f8fafc')

for ax, mat, xlabs, title in [
    (ax1, m_ch, CHANNELS,  'Algorithm × Channel (SGD 500k)'),
    (ax2, m_am, AMT_LABS,  'Algorithm × Amount (SWIFT_MT103)'),
]:
    im = ax.imshow(mat, cmap=risk_cmap, vmin=0, vmax=100, aspect='auto')
    ax.set_xticks(range(len(xlabs))); ax.set_xticklabels(xlabs, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(len(ALGORITHMS))); ax.set_yticklabels(ALGORITHMS, fontsize=9)
    ax.set_title(title, fontsize=11)
    fig.colorbar(im, ax=ax, label='Score / 100')
    for i in range(len(ALGORITHMS)):
        for j in range(len(xlabs)):
            ax.text(j, i, f'{mat[i,j]:.0f}', ha='center', va='center', fontsize=8,
                    color='white' if mat[i,j] < 55 else '#101828')

plt.tight_layout(); plt.show()

## Interpretation Summary

| Algorithm    | Quantum Safe | NIST Standard | Score drop drivers |
|---|---|---|---|
| RSA-2048     | No  | Classical legacy | Algorithm penalty (−20) + exposure |
| ML-KEM-768   | Yes | FIPS 203         | Only exposure-based deductions |
| ML-DSA-65    | Yes | FIPS 204         | Only exposure-based deductions |
| SLH-DSA-128s | Yes | FIPS 205         | Only exposure-based deductions |
| HQC-128      | Yes | 2025 selected    | Only exposure-based deductions |

The heatmap shows RSA-2048 scores drop sharply on SWIFT/WIRE channels at high amounts.
PQC algorithms maintain high scores across all channels — confirming migration is urgent for cross-border SWIFT payments.